# TabTune · Enterprise Time Series for Banking
### Liquidity forecasting and anomaly detection with time-series foundation models

**Library:** TabTune open source (`tabtune.TimeSeries`) · **Runtime:** Google Colab Pro+ with a GPU (A100 or L4)

| Part | What you will see |
|---|---|
| 1 | A 20-series GCC banking panel: branch cash, corporate payment outflows (GTB), card spend (meem) |
| 2 | Exploratory dashboard: seasonality, Ramadan and Eid effects, payday spikes |
| 3 | Model race: foundation models vs baselines on rolling backtests |
| 4 | 4-week liquidity forecast with calendar covariates and calibrated bands |
| 5 | Executive liquidity dashboard |
| 6 | Ensembles and series similarity from embeddings |
| 7 | Anomaly detection on a live feed: detectors, scoring against ground truth, alert dashboard |

> **Data note.** The panel is synthetic and generated in this notebook. It reproduces GCC banking calendar effects (Friday–Saturday weekend, Ramadan, Eid al-Fitr and al-Adha, the 27th-of-month government payroll, national days) and contains no customer data. Anomalies are injected with known labels so detectors can be scored honestly.

**Before you start:** *Runtime → Change runtime type → GPU (A100 or L4)*, then *Runtime → Run all*. First run downloads model weights (about 1–2 GB).

In [ ]:
# ⚙️ 1. Install TabTune (open source) on Colab — about 2–3 minutes
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tabtune[colab] @ git+https://github.com/Lexsi-Labs/TabTune.git", "plotly"], check=True)
print("Colab:", IN_COLAB)

In [ ]:
# 🔧 Demo configuration
import logging, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from tabtune.logger import setup_logger
from tabtune.TimeSeries import (TimeSeriesPipeline, TimeSeriesSchema, TimeSeriesLeaderboard,
                                TimeSeriesEnsemble, split_horizon)

warnings.filterwarnings("ignore")
setup_logger(level=logging.WARNING)          # keep the notebook output clean
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# FAST=1 swaps foundation models for weight-free models (smoke tests without a GPU or downloads)
FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"

HORIZON = 28                                  # 4-week liquidity horizon (daily data)
QUANTILES = [0.05, 0.1, 0.5, 0.9, 0.95]
FOUNDATION_MODELS = [] if FAST else ["ChronosBolt", "Chronos2"]   # Apache-2.0 weights: commercial-safe
BASELINES = ["SeasonalNaive", "WindowAverage", "TabularTS-GBM"]
PRIMARY = "TabularTS-GBM" if FAST else "Chronos2"   # reads calendar covariates
DETECTOR = "SeasonalNaive" if FAST else "ChronosBolt"
EMBEDDER = None if FAST else "ChronosBolt"
RUN_FINETUNE = False                          # set True for the optional LoRA section (about 5 min on A100)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
BLUE, ORANGE, GREY, DARK = "#2F5BEA", "#B4541A", "#9AA0A8", "#16181D"
SEG_COLORS = {"Retail branch cash": BLUE, "Corporate payments (GTB)": ORANGE, "Card spend (meem)": "#1B998B"}
print(f"Device: {DEVICE} · primary model: {PRIMARY} · detector: {DETECTOR} · fast mode: {FAST}")

## 1 · A GCC banking panel

Twenty daily series from January 2022 to October 2026, in USD millions:

* **Retail branch cash** (8 branches across KSA, Bahrain, UAE): Thursday peaks, Friday lows, payday and pre-Eid surges.
* **Corporate payments (GTB)** (8 industry clusters): weekday-only flows, payroll and month-end and quarter-end peaks, Eid shutdowns.
* **Card spend (meem)** (4 markets): fast digital growth, Ramadan and pre-Eid uplift.

The calendar becomes **known-in-advance covariates**, which foundation models such as Chronos-2 can read.

In [ ]:
# 🏦 Generate the panel and the GCC calendar
RAMADAN = [("2022-04-02", "2022-05-01"), ("2023-03-23", "2023-04-20"), ("2024-03-11", "2024-04-09"),
           ("2025-03-01", "2025-03-29"), ("2026-02-18", "2026-03-19")]      # approximate (moon sighting)
EID_FITR = ["2022-05-02", "2023-04-21", "2024-04-10", "2025-03-30", "2026-03-20"]
EID_ADHA = ["2022-07-09", "2023-06-28", "2024-06-16", "2025-06-06", "2026-05-27"]
DATES = pd.date_range("2022-01-01", "2026-10-31", freq="D")


def gcc_calendar(dates):
    cal = pd.DataFrame({"timestamp": dates})
    dow = dates.dayofweek
    cal["is_weekend"] = np.isin(dow, [4, 5]).astype(int)                 # Friday–Saturday
    cal["is_payday"] = ((dates.day >= 26) & (dates.day <= 28)).astype(int)  # govt payroll around the 27th
    cal["is_month_end"] = (dates.is_month_end | (dates + pd.Timedelta(days=1)).is_month_end).astype(int)
    ram, pre, eid = (np.zeros(len(dates), bool) for _ in range(3))
    for s, e in RAMADAN:
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        ram |= (dates >= s) & (dates <= e)
        pre |= (dates > e - pd.Timedelta(days=7)) & (dates <= e)
    for d in EID_FITR:
        d = pd.Timestamp(d); eid |= (dates >= d) & (dates < d + pd.Timedelta(days=4))
    for d in EID_ADHA:
        d = pd.Timestamp(d); eid |= (dates >= d - pd.Timedelta(days=1)) & (dates < d + pd.Timedelta(days=4))
    national = ((dates.month == 9) & (dates.day == 23)) | ((dates.month == 12) & (dates.day == 16)) | \
               ((dates.month == 2) & (dates.day == 22))
    cal["is_ramadan"], cal["is_pre_eid"], cal["is_eid"] = ram.astype(int), pre.astype(int), eid.astype(int)
    cal["is_holiday"] = (eid | national).astype(int)
    return cal


SERIES = (
    [("Retail branch cash", f"BR-{c}", c, base) for c, base in
     [("Riyadh", 6.0), ("Jeddah", 4.8), ("Dammam", 3.1), ("Khobar", 2.6), ("Manama", 2.9),
      ("Muharraq", 1.6), ("Abu Dhabi", 2.2), ("Dubai", 3.4)]]
    + [("Corporate payments (GTB)", f"GTB-{c}", "Group", base) for c, base in
       [("Energy", 120), ("Construction", 64), ("Trade finance", 85), ("Government", 140),
        ("Healthcare", 38), ("Retail chains", 52), ("Logistics", 30), ("Telecom", 45)]]
    + [("Card spend (meem)", f"CARD-{c}", c, base) for c, base in
       [("KSA", 14.0), ("Bahrain", 4.0), ("UAE", 6.5), ("Kuwait", 3.2)]]
)


def make_gcc_bank_panel(seed=7):
    rng = np.random.default_rng(seed)
    cal = gcc_calendar(DATES)
    t = np.arange(len(DATES)) / 365.25
    dow, month, day = DATES.dayofweek.values, DATES.month.values, DATES.day.values
    rows, meta = [], []
    for segment, item, region, base in SERIES:
        if segment == "Retail branch cash":
            weekly = np.select([dow == 3, dow == 4, dow == 5], [1.15, 0.65, 0.90], 1.0)
            level = (base * (1.06 ** t) * weekly * (1 + 0.45 * cal.is_payday) * (1 + 0.20 * cal.is_ramadan)
                     * (1 + 0.55 * cal.is_pre_eid) * (1 - 0.55 * cal.is_eid) * (1 + 0.08 * (month == 6)))
            noise = 0.08
        elif segment == "Corporate payments (GTB)":
            weekly = np.where(np.isin(dow, [4, 5]), 0.08, 1.0)
            payroll = (day >= 24) & (day <= 26)
            quarter_end = np.isin(month, [3, 6, 9, 12]) & (day >= 28)
            level = (base * (1.08 ** t) * weekly * (1 + 0.8 * payroll) * (1 + 0.6 * cal.is_month_end)
                     * (1 + 0.4 * quarter_end) * (1 - 0.12 * cal.is_ramadan) * (1 - 0.85 * cal.is_eid))
            noise = 0.15
        else:
            weekly = np.select([dow == 3, dow == 4], [1.12, 1.12], 1.0)
            level = (base * (1.15 ** t) * weekly * (1 + 0.30 * cal.is_payday) * (1 + 0.18 * cal.is_ramadan)
                     * (1 + 0.70 * cal.is_pre_eid) * (1 - 0.20 * cal.is_eid))
            noise = 0.06
        values = np.asarray(level) * rng.lognormal(0, noise, len(DATES))
        rows.append(pd.DataFrame({"item_id": item, "timestamp": DATES, "cash": values}))
        meta.append({"item_id": item, "segment": segment, "region": region})
    df = pd.concat(rows, ignore_index=True).merge(cal, on="timestamp")
    return df, pd.DataFrame(meta)


df, meta = make_gcc_bank_panel()
COVARIATES = ["is_weekend", "is_payday", "is_month_end", "is_ramadan", "is_pre_eid", "is_eid", "is_holiday"]
print(f"{df.item_id.nunique()} series × {df.timestamp.nunique()} days = {len(df):,} rows")
display(df.head())
display(meta.groupby("segment").size().rename("series").to_frame())

## 2 · Exploratory dashboard

In [ ]:
# 📊 Segment totals, weekly profile and Ramadan effect
seg = df.merge(meta, on="item_id")
totals = seg.groupby(["segment", pd.Grouper(key="timestamp", freq="W")])["cash"].sum().reset_index()

fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2, "secondary_y": True}, None], [{}, {}]], vertical_spacing=0.14,
                    subplot_titles=("Weekly flows by segment (USD mn)", "Average day-of-week profile (index)",
                                    "Ramadan, pre-Eid and Eid effect vs normal days"))
for s, g in totals.groupby("segment"):
    fig.add_trace(go.Scatter(x=g.timestamp, y=g.cash, name=s + (" (right axis)" if "GTB" in s else ""),
                             line=dict(color=SEG_COLORS[s], width=2)), 1, 1, secondary_y="GTB" in s)
for s0, e0 in RAMADAN:
    fig.add_vrect(x0=s0, x1=e0, fillcolor="#E8C547", opacity=0.18, line_width=0, row=1, col=1)

names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dowp = seg.assign(dow=seg.timestamp.dt.dayofweek).groupby(["segment", "dow"])["cash"].mean()
dowp = (dowp / dowp.groupby(level=0).transform("mean")).reset_index()
for s, g in dowp.groupby("segment"):
    fig.add_trace(go.Bar(x=[names[d] for d in g.dow], y=g.cash, name=s, marker_color=SEG_COLORS[s],
                         showlegend=False), 2, 1)

state = np.select([seg.is_eid == 1, seg.is_pre_eid == 1, seg.is_ramadan == 1], ["Eid", "Pre-Eid", "Ramadan"], "Normal")
eff = seg.assign(state=state).groupby(["segment", "state"])["cash"].mean().unstack()
eff = (eff.div(eff["Normal"], axis=0) - 1) * 100
for s in eff.index:
    fig.add_trace(go.Bar(x=["Ramadan", "Pre-Eid", "Eid"], y=eff.loc[s, ["Ramadan", "Pre-Eid", "Eid"]],
                         marker_color=SEG_COLORS[s], showlegend=False), 2, 2)
fig.update_yaxes(title_text="% vs normal", row=2, col=2)
fig.update_yaxes(title_text="retail, card", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="corporate", row=1, col=1, secondary_y=True, showgrid=False)
fig.update_layout(template="plotly_white", height=760, barmode="group",
                  title="GCC banking panel · shaded bands are Ramadan", legend=dict(orientation="h", y=1.08))
fig.show()

## 3 · Model race on rolling backtests

`TimeSeriesLeaderboard` fits each model and scores it on the last three 4-week windows of history, never on data it was fitted on. Foundation models run **zero-shot**: no training on this bank's data. Classic baselines set the floor.

Metrics: **MASE** (error relative to a seasonal-naive forecast; below 1 beats it) and **WQL** (quantile loss: how good the whole forecast distribution is).

Each series is modelled as an **index of its own last-year average**, so errors are comparable between a USD 1.6 mn branch and a USD 140 mn corporate cluster. Forecasts are converted back to USD for the dashboards.

In [ ]:
# 🏁 Hold out the final 4 weeks, race the models on the rest
schema = TimeSeriesSchema(target="cash", item_id="item_id")
schema_cov = TimeSeriesSchema(target="cash", item_id="item_id", known_covariates=COVARIATES)
history_usd, actual_usd = split_horizon(df, schema_cov, prediction_length=HORIZON)
SCALE = history_usd.groupby("item_id")["cash"].apply(lambda s: s.tail(365).mean()).rename("scale")
to_index = lambda d: d.assign(cash=d.cash / d.item_id.map(SCALE))
history, actual = to_index(history_usd), to_index(actual_usd)
future_cov = actual[["item_id", "timestamp", *COVARIATES]]
history_y = history[["item_id", "timestamp", "cash"]]
fp = {"prediction_length": HORIZON, "quantile_levels": QUANTILES}

board = TimeSeriesLeaderboard(history_y, schema, forecast_params=fp, windows=3)
board.add_models(BASELINES + FOUNDATION_MODELS)
t0 = time.time()
ranking = board.run(rank_by="mase", display=False)
print(f"Leaderboard finished in {time.time() - t0:.0f}s")
display(ranking)

In [ ]:
# 📈 Leaderboard chart
ok = ranking[ranking["Status"].eq("ok")] if "Status" in ranking else ranking
kind = ["Foundation model" if m in FOUNDATION_MODELS else "Baseline" for m in ok["Model"]]
fig = make_subplots(rows=1, cols=2, subplot_titles=("MASE (lower is better)", "WQL (lower is better)"))
for i, metric in enumerate(["mase", "wql"], start=1):
    order = ok.sort_values(metric)
    fig.add_trace(go.Bar(y=order["Model"], x=order[metric], orientation="h",
                         marker_color=[BLUE if m in FOUNDATION_MODELS else GREY for m in order["Model"]],
                         text=order[metric].round(3), textposition="outside", showlegend=False), 1, i)
fig.add_vline(x=1.0, line_dash="dot", line_color=DARK, row=1, col=1)
fig.update_layout(template="plotly_white", height=420,
                  title="Rolling-backtest leaderboard · blue = foundation model (zero-shot), grey = baseline")
fig.show()

## 4 · Four-week liquidity forecast with calendar covariates

The primary model reads the GCC calendar as **known-in-advance covariates**: weekends, payday, month-end, Ramadan, pre-Eid, Eid and national holidays. We compare it with the same model without covariates, then calibrate the intervals with conformal prediction so a 90% band really holds about 90% of outcomes.

In [ ]:
# 🔮 Forecast with and without the calendar
plain = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial",
                           tuning_params={"device": DEVICE})
plain.fit(history_y, schema)
m_plain = plain.evaluate(actual)

pipe = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial",
                          tuning_params={"device": DEVICE})
pipe.fit(history, schema_cov, future_df=future_cov)
forecast = pipe.predict()
m_cov = pipe.evaluate(actual, forecast=forecast)

KEYS = [k for k in ("mase", "wql", "smape", "coverage_80", "coverage_90") if k in m_cov]
comparison = pd.DataFrame({"target only": m_plain, "with GCC calendar": m_cov}).loc[KEYS]
display(comparison)
gain = (1 - m_cov["mase"] / m_plain["mase"]) * 100
display(Markdown(f"**Calendar covariates change MASE by {gain:+.1f}%** on the 4-week holdout ({PRIMARY})."))

In [ ]:
# 🎯 Conformal calibration of the intervals
pipe.calibrate(windows=4)
forecast_cal = pipe.predict()
m_cal = pipe.evaluate(actual, forecast=forecast_cal)
cal_table = pd.DataFrame({"raw": m_cov, "calibrated": m_cal}).loc[[k for k in KEYS if k.startswith("coverage")] + ["wql", "mase"]]
display(cal_table)
fc = forecast_cal.to_pandas()
QCOLS = ["point"] + [str(q) for q in QUANTILES]
fc[QCOLS] = fc[QCOLS].mul(fc.item_id.map(SCALE), axis=0)          # back to USD mn
print("Forecast columns:", list(fc.columns))

In [ ]:
# 🌀 Interactive fan chart: pick a series from the dropdown
hist_tail = history_usd[history_usd.timestamp > history_usd.timestamp.max() - pd.Timedelta(days=120)]
items = list(meta.item_id)
fig = go.Figure()
for k, item in enumerate(items):
    h = hist_tail[hist_tail.item_id == item]; a = actual_usd[actual_usd.item_id == item]; f = fc[fc.item_id == item]
    vis = k == 0
    fig.add_trace(go.Scatter(x=h.timestamp, y=h.cash, name="history", line=dict(color=DARK, width=1.5), visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.95"], f["0.05"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.12)", line=dict(width=0), name="90% band", visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.9"], f["0.1"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.25)", line=dict(width=0), name="80% band", visible=vis))
    fig.add_trace(go.Scatter(x=f.timestamp, y=f["0.5"], name="median forecast", line=dict(color=BLUE, width=2.5), visible=vis))
    fig.add_trace(go.Scatter(x=a.timestamp, y=a.cash, name="actual", mode="markers",
                             marker=dict(color=ORANGE, size=6), visible=vis))
n = 5
buttons = [dict(label=item, method="update",
                args=[{"visible": [j // n == k for j in range(n * len(items))]}, {"title": f"{item} · 4-week forecast"}])
           for k, item in enumerate(items)]
fig.update_layout(template="plotly_white", height=520, title=f"{items[0]} · 4-week forecast",
                  updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.15, direction="down")],
                  yaxis_title="USD mn per day", legend=dict(orientation="h", y=-0.15))
fig.show()

## 5 · Executive liquidity dashboard

In [ ]:
# 🧭 KPIs and segment-level outlook for the next 4 weeks
fcs = fc.merge(meta, on="item_id")
need = fcs.groupby("segment")[["0.5", "0.95"]].sum()        # band = sum of series quantiles (conservative)
per_item = fcs.merge(actual_usd[["item_id", "timestamp", "cash"]], on=["item_id", "timestamp"])
per_item["inside_90"] = per_item.cash.between(per_item["0.05"], per_item["0.95"])

fig = make_subplots(rows=2, cols=4, row_heights=[0.32, 0.68], vertical_spacing=0.12,
                    specs=[[{"type": "indicator"}] * 4, [{"colspan": 2, "secondary_y": True}, None, {"colspan": 2}, None]],
                    subplot_titles=("", "", "", "", "Daily outflow outlook by segment (median, USD mn)",
                                    "4-week requirement: median vs 95th percentile (USD mn)"))
kpis = [("4-week outflow<br>median, USD mn", need["0.5"].sum()),
        ("Stress buffer<br>to P95, USD mn", need["0.95"].sum() - need["0.5"].sum()),
        ("Forecast error<br>holdout MASE", m_cal["mase"]),
        ("Outcomes inside<br>90% band", per_item.inside_90.mean() * 100)]
for i, (title, value) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(mode="number", value=value, title={"text": title},
                               number={"valueformat": ",.2f" if value < 50 else ",.0f",
                                       "suffix": "%" if "band" in title else "", "font": {"size": 48}}), 1, i)
daily = fcs.groupby(["segment", "timestamp"])["0.5"].sum().reset_index()
for s, g in daily.groupby("segment"):
    fig.add_trace(go.Scatter(x=g.timestamp, y=g["0.5"], name=s + (" (right axis)" if "GTB" in s else ""),
                             line=dict(color=SEG_COLORS[s], width=2.5)), 2, 1, secondary_y="GTB" in s)
fig.add_trace(go.Bar(x=need.index, y=need["0.5"], name="median", marker_color=BLUE), 2, 3)
fig.add_trace(go.Bar(x=need.index, y=need["0.95"] - need["0.5"], name="buffer to P95", marker_color=ORANGE), 2, 3)
fig.update_layout(template="plotly_white", height=720, barmode="stack",
                  title=f"Liquidity outlook · {fc.timestamp.min():%d %b} – {fc.timestamp.max():%d %b %Y} · model {PRIMARY}",
                  legend=dict(orientation="h", y=-0.08))
fig.show()

## 6 · Ensembles and series similarity

`TimeSeriesEnsemble` combines models with greedy selection on backtests. Embeddings from a foundation model group series that behave alike, which is useful for segment-level models and for spotting a branch that starts behaving like another segment.

In [ ]:
# 🤝 Greedy ensemble
members = BASELINES + FOUNDATION_MODELS
ensemble = TimeSeriesEnsemble(members, ensemble_strategy="greedy_selection", forecast_params=fp, verbose=False)
ensemble.fit(history_y, schema)
display(ensemble.get_leaderboard())
m_ens = ensemble.evaluate(actual)
display(pd.Series({k: m_ens[k] for k in ("mase", "wql") if k in m_ens}, name="ensemble holdout").to_frame())

In [ ]:
# 🧬 Series similarity from foundation-model embeddings
if EMBEDDER:
    emb = TimeSeriesPipeline(EMBEDDER, task_type="embedding", tuning_params={"device": DEVICE}).fit(history_y, schema).predict()
    sim = pd.DataFrame(emb.similarity(), index=list(emb.item_ids), columns=list(emb.item_ids))
    order = meta.sort_values("segment").item_id
    sim = sim.loc[order, order]
    fig = px.imshow(sim, color_continuous_scale="Blues", zmin=sim.values.min(), zmax=1,
                    title=f"Cosine similarity of {EMBEDDER} embeddings · {emb.dim}-dim per series")
    fig.update_layout(height=640, template="plotly_white")
    fig.show()
else:
    print("Embeddings need a foundation model (skipped in fast mode).")

In [ ]:
# 🛠️ Optional: LoRA fine-tuning vs zero-shot (set RUN_FINETUNE = True in the config cell)
if RUN_FINETUNE and not FAST:
    tuned = TimeSeriesPipeline("ChronosBolt", tuning_strategy="peft", forecast_params=fp,
                               tuning_params={"epochs": 2, "steps_per_epoch": 50, "device": DEVICE})
    tuned.fit(history_y, schema)
    zero = TimeSeriesPipeline("ChronosBolt", forecast_params=fp, tuning_params={"device": DEVICE}).fit(history_y, schema)
    display(pd.DataFrame({"zero-shot": zero.evaluate(actual), "LoRA, 2 epochs": tuned.evaluate(actual)}).loc[["mase", "wql"]])
else:
    print("Skipped. Set RUN_FINETUNE = True to compare LoRA fine-tuning with zero-shot.")

## 7 · Anomaly detection on a live feed

We take the last 480 days of the panel as a **live monitoring feed** and inject labelled incidents into its final 150 days:

| Incident | What it looks like | Typical cause |
|---|---|---|
| Spike | one day at 3–5× normal | suspected fraud, large one-off withdrawal |
| Outage | 1–3 days near zero | ATM or channel outage, failed data feed |
| Level shift | 5–10 days at ±40–60% | system migration, reporting break |

TabTune scores each day against rolling forecasts made **only from the history before it**, converts scores into conformal p-values against a clean reference period, and flags days below the false-alarm rate `alpha`.

In [ ]:
# 🚨 Build the live feed with labelled incidents
rng = np.random.default_rng(11)
LIVE_DAYS, INJECT_DAYS = 480, 150
live = df[df.timestamp > df.timestamp.max() - pd.Timedelta(days=LIVE_DAYS)][["item_id", "timestamp", "cash"]].copy()
live = live.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
live["is_anomaly"] = False
events = []
cutoff = live.timestamp.max() - pd.Timedelta(days=INJECT_DAYS)
for item, idx in live.groupby("item_id").indices.items():
    idx = np.asarray(idx)
    window = idx[live.loc[idx, "timestamp"].values > cutoff]
    used = set()
    for kind in rng.choice(["spike", "spike", "outage", "level_shift"], size=rng.integers(3, 6)):
        length = {"spike": 1, "outage": int(rng.integers(1, 4)), "level_shift": int(rng.integers(5, 11))}[kind]
        for _ in range(20):
            start = int(rng.integers(0, len(window) - length))
            span = set(range(start, start + length))
            if not span & used:
                break
        used |= span
        rows = window[start:start + length]
        factor = {"spike": rng.uniform(3, 5), "outage": rng.uniform(0.02, 0.15),
                  "level_shift": rng.choice([rng.uniform(1.4, 1.6), rng.uniform(0.4, 0.6)])}[kind]
        live.loc[rows, "cash"] *= factor
        live.loc[rows, "is_anomaly"] = True
        events.append({"item_id": item, "start": live.loc[rows[0], "timestamp"], "days": length,
                       "type": kind, "factor": round(float(factor), 2)})
events = pd.DataFrame(events).sort_values(["item_id", "start"]).reset_index(drop=True)
labels = live[["item_id", "timestamp", "is_anomaly"]]
feed = live.drop(columns="is_anomaly")
print(f"{len(events)} incidents injected · {int(labels.is_anomaly.sum())} anomalous days out of {len(live):,}")
display(events.groupby("type").agg(incidents=("item_id", "size"), days=("days", "sum")))

In [ ]:
# 🔍 Three detection methods on the same forecaster, scored against the labels
TASK = {"min_context": 240, "alpha": 0.02, "reference_fraction": 0.3}
results, scores = {}, []
for method in ("interval", "forecast_error", "likelihood"):
    t0 = time.time()
    det = TimeSeriesPipeline(DETECTOR, task_type="anomaly_detection", tuning_params={"device": DEVICE},
                             task_params={"method": method, **TASK})
    res = det.fit(feed, schema).predict()
    results[method] = res
    scores.append({"method": method, **res.evaluate(labels), "seconds": round(time.time() - t0, 1)})
scores = pd.DataFrame(scores).set_index("method")


def rolling_median_score(values):          # classic robust baseline (looks at both sides of each day)
    med = values.rolling(29, center=True, min_periods=7).median()
    spread = (values - med).abs().rolling(29, center=True, min_periods=7).median()
    return (values - med).abs() / (1.4826 * spread + 1e-9)


from sklearn.metrics import roc_auc_score, average_precision_score
scored = results["interval"].to_pandas()[["item_id", "timestamp", "score"]].dropna()
base = feed.assign(base=feed.groupby("item_id")["cash"].transform(rolling_median_score))
chk = scored.merge(base, on=["item_id", "timestamp"]).merge(labels, on=["item_id", "timestamp"])
scores.loc["rolling median (baseline)", ["auroc", "auprc"]] = [roc_auc_score(chk.is_anomaly, chk.base),
                                                                 average_precision_score(chk.is_anomaly, chk.base)]
display(scores[["n", "n_anomalies", "auroc", "auprc", "precision", "recall", "f1", "seconds"]])
BEST = scores.drop("rolling median (baseline)")["auprc"].idxmax()
display(Markdown(f"**Best detector by AUPRC: `{BEST}`** with {DETECTOR}. Always compare against the simple baseline."))

In [ ]:
# 🖥️ Monitoring dashboard: expected band, flagged days and true incidents (pick a series)
res = results[BEST].to_pandas().merge(labels, on=["item_id", "timestamp"], how="left")
items = list(meta.item_id)
fig = go.Figure()
per = 5
for k, item in enumerate(items):
    r = res[res.item_id == item]; f = feed[feed.item_id == item]; vis = k == 0
    fig.add_trace(go.Scatter(x=pd.concat([r.timestamp, r.timestamp[::-1]]), y=pd.concat([r.upper, r.lower[::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.14)", line=dict(width=0),
                             name="expected range", visible=vis))
    fig.add_trace(go.Scatter(x=f.timestamp, y=f.cash, name="observed", line=dict(color=DARK, width=1.3), visible=vis))
    t = r[r.is_anomaly_y.fillna(False).astype(bool)]
    fig.add_trace(go.Scatter(x=t.timestamp, y=t.value, mode="markers", name="true incident",
                             marker=dict(symbol="circle-open", size=14, color=ORANGE, line=dict(width=2)), visible=vis))
    a = r[r.is_anomaly_x.fillna(False).astype(bool)]
    fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, mode="markers", name="TabTune alert",
                             marker=dict(symbol="x", size=9, color="#D7263D"), visible=vis))
    fig.add_trace(go.Scatter(x=r.timestamp, y=r.score, name="anomaly score", yaxis="y2",
                             line=dict(color=GREY, width=1), visible=vis))
buttons = [dict(label=item, method="update",
                args=[{"visible": [j // per == k for j in range(per * len(items))]}, {"title": f"{item} · live monitoring"}])
           for k, item in enumerate(items)]
fig.update_layout(template="plotly_white", height=560, title=f"{items[0]} · live monitoring",
                  yaxis=dict(title="USD mn per day", domain=[0.3, 1]), yaxis2=dict(title="score", domain=[0, 0.22]),
                  updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.15)],
                  legend=dict(orientation="h", y=-0.12))
fig.show()

In [ ]:
# 📋 Alert queue for the operations team, and detection rate by incident type
alerts = results[BEST].anomalies.merge(meta, on="item_id")
alerts["deviation_%"] = (alerts.value / alerts.expected - 1) * 100
alerts = alerts.merge(labels.rename(columns={"is_anomaly": "confirmed_incident"}), on=["item_id", "timestamp"])
queue = alerts.sort_values("score", ascending=False)[
    ["timestamp", "item_id", "segment", "value", "expected", "deviation_%", "p_value", "confirmed_incident"]].head(15)
display(queue.style.format({"value": "{:,.2f}", "expected": "{:,.2f}", "deviation_%": "{:+.0f}%", "p_value": "{:.4f}"})
        .background_gradient(subset=["deviation_%"], cmap="RdBu_r", vmin=-100, vmax=100)
        .set_caption("Top 15 alerts by anomaly score"))

flagged = set(zip(alerts.item_id, alerts.timestamp))
caught = []
for _, e in events.iterrows():
    days = pd.date_range(e.start, periods=e.days, freq="D")
    caught.append(any((e.item_id, d) in flagged for d in days))
events["detected"] = caught
by_type = events.groupby("type")["detected"].mean().mul(100).reset_index()
weeks = pd.date_range(feed.timestamp.max() - pd.Timedelta(days=INJECT_DAYS + 60), feed.timestamp.max(), freq="W")
weekly = (alerts.groupby(["item_id", pd.Grouper(key="timestamp", freq="W")]).size().unstack(fill_value=0)
          .reindex(index=list(meta.item_id), columns=weeks, fill_value=0))

fig = make_subplots(rows=1, cols=2, column_widths=[0.35, 0.65],
                    subplot_titles=("Incidents detected by type (%)", "Alerts per series per week"))
fig.add_trace(go.Bar(x=by_type.type, y=by_type.detected, marker_color=[BLUE, ORANGE, "#1B998B"][:len(by_type)],
                     text=by_type.detected.round(0), textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=weekly.values, x=weekly.columns, y=weekly.index, colorscale=[[0, "#F7F7F4"], [1, ORANGE]], zmin=0, showscale=True, colorbar=dict(title="alerts")), 1, 2)
fig.update_yaxes(range=[0, 110], row=1, col=1)
fig.update_layout(template="plotly_white", height=560, title=f"Alert analytics · {DETECTOR} · method {BEST}")
fig.show()
print(f"Incidents detected: {events.detected.sum()} of {len(events)} · alerts raised: {len(alerts)} · "
      f"alerts on confirmed incident days: {alerts.confirmed_incident.mean():.0%}")

## Summary

| Capability | TabTune call | Shown with |
|---|---|---|
| Multi-series forecasting, zero-shot | `TimeSeriesPipeline(model).fit().predict()` | 20 series, 4-week horizon |
| Known-in-advance covariates | `TimeSeriesSchema(known_covariates=...)`, `future_df` | GCC calendar: Ramadan, Eid, payday |
| Model race on rolling backtests | `TimeSeriesLeaderboard` | foundation models vs baselines |
| Calibrated probabilistic forecasts | `calibrate()` | conformal 80% and 90% bands |
| Ensembles | `TimeSeriesEnsemble(greedy_selection)` | baselines + foundation models |
| Embeddings | `task_type="embedding"` | series similarity map |
| Anomaly detection | `task_type="anomaly_detection"` | live feed, scored against labelled incidents |
| Licence safety | `license_mode="commercial"` | Chronos-2 and Chronos-Bolt are Apache-2.0 |

Swap the synthetic panel for a bank's own daily series (same long format: `item_id`, `timestamp`, value, calendar columns) and every cell runs unchanged.